In [ ]:
import pandas as pd
import json
import re
from string import Template
from enum import Enum
from pydantic import BaseModel, Field
from typing import List, TypeVar, Generic
from docx import Document
from docx.oxml.ns import qn
import json
import requests
import os
from striprtf.striprtf import rtf_to_text

In [2]:
class HFrEF_Etiology(str, Enum):
    ISCHEMIC = "Ischemic"
    NON_ISCHEMIC = "Non-ischemic"

class Drug_Class(str, Enum):
    ACE_I = "ACE-I"
    ARB = "ARB"
    ARNI = "ARNI"
    BETA_BLOCKER = "Beta-blocker"
    MRA = "MRA"
    SGLT2_INHIBITOR = "SGLT2 Inhibitor"
    LOOP_DIURETIC = "Loop Diuretic"
    OTHER = "Other"

## Original variables

In [ ]:
class TherapyEntry(BaseModel):
    drug_name: str = Field(default=None,
        description="Drug name as reported in the source, exactly as written. "
                    "Always populate this field."
    )
    generic_drug_name: str = Field(default=None,
        description="Drug generic name or ATC code, only if clearly identifiable as such. "
                    "Return the output translated in ENGLISH"
                    "Exclude the dose from the output. "
                    "Set to null if not reported or not distinguishable."
    )
    brand_drug_name: str = Field(default=None,
        description="Drug brand name, only if clearly identifiable as such. "
                    "Set to null if not reported or not distinguishable."
    )
    active_ingredient: str = Field(default=None,
        description="Extract the dose strength and its mass/activity unit of measurement (e.g., mg, mcg, µg, UI) for the medication, as explicitly reported in the text."
                    "Formatting rules:"
                    "- For single active ingredients: Output MUST be strictly in the format <number> <unit> (e.g., 20 mg, 50 mcg, 1000 UI)."
                    "- For combination drugs (e.g., 97mg+103mg or 97mg/103mg): Join the numbers with a slash and follow with a single space and unit, formatted as <number>/<number> <unit> (e.g., 97/103 mg). If units differ, maintain their respective units (e.g., 50 mcg/10 mg)."
                    "Ignore pharmaceutical forms, tablet/pill counts, administration timing, and frequencies."
    )

    dose_units_number: float = Field(default=None,
        description="Calculate ONLY the total count of single physical dose units (e.g., tablets, drops, vials, puffs) to be taken across the ENTIRE 24-HOUR DAY, expressed as a single decimal or integer number."
                    "Rules:"
                    "MULTIPLE TIMESTAMPS FOR ONE DOSE: If a dose is linked to multiple hours separated by hyphens, commas, or slashes (e.g., '1 cpr at 8-20', '1/2 cpr hour 8,14,20'), multiply the unit dose by the number of listed hours (e.g., 1 tablet at 8 and 20 = 1 * 2 = 2)."
                    "- SUM MULTIPLE EVENTS: If multiple individual doses, timestamps, or times of day are listed (e.g., '1 tablet at 8:00 and 1 tablet at 20:00', '1 tab morning and 1 tab night'), SUM all individual physical units together (1 + 1 = 2)."
                    "- FREQUENCY MULTIPLICATION: If an explicit daily frequency is stated (e.g., '1/2 tablet twice a day', '1 puff 3 times daily'), multiply the unit dose by the frequency (0.5 * 2 = 1)."
                    "- TIMESTAMPS VS COUNTS: Do NOT use the hour value as a dose (e.g., in 'at 14:00', 14 is the hour, NOT 14 tablets). However, DO count the physical dose associated with that time."
                    "- CONVERSIONS: Convert words, fractions, and partial units to decimals (e.g., 1/2 -> 0.5, half -> 0.5, one -> 1, two -> 2)."
                    "- IGNORE DOSAGE STRENGTHS: Ignore mass/volume units (e.g., mg, mcg, ml) and focus ONLY on physical unit counts."
                    "- SINGLE EVENT: If only a single administration is described for the day with no other times mentioned (e.g., 1/2 tablet at 14), the daily total is that single dose (0.5)."

                    "Examples: "
                    "- '1 cpr at 8-20' -> 2"
                    "- '1/2 cpr at 8-14-20' -> 1.5"
                    "- '1/2 tablet twice a day' -> 1"
                    "- '1 tablet morning, 1/2 tablet at 14:00, 1 tablet at night' -> 2.5"
                    "- 'Bisoprolol 5mg 1/2 tablet at 8:00' -> 0.5"

                    "Output MUST be ONLY the single calculated number or null. No text, explanation, or unit labels."
    )
    
    drug_class: Drug_Class = Field(default=None,
        description="Analyze the drug name following these steps:"
                    "1. Identify the active ingredient."
                    "2. Assign the corresponding class from the predefined list: ACE_I / ARB / ARNI / BETA_BLOCKER / MRA / SGLT2_INHIBITOR / LOOP_DIURETIC."
                    "Examples:"
                    "- ACE_I: enalapril, ramipril, lisinopril"
                    "- ARB: losartan, valsartan, candesartan"
                    "- ARNI: sacubitril/valsartan"
                    "- BETA_BLOCKER: bisoprolol, carvedilol, metoprolol"
                    "- MRA: spironolactone, eplerenone, canrenone (and its potassium salt, "
                    "  canrenoate)"
                    "- SGLT2_INHIBITOR: dapagliflozin, empagliflozin"
                    "- LOOP_DIURETIC: furosemide, torasemide, bumetanide"
                    "3. If the active ingredient does not match any list, set to null."
    )
    notes: str = Field(default=None,
        description="Extra information about the specified drug. "
                    "Specify any relevant details such as dose modifications, uptitration status, dates of therapy initiation or changes, or other clinically relevant information. "
                    "Set to null if not reported."
    )

class LabEntry(BaseModel):
    lab_exam: str = Field(default=None,
        description="Name of the laboratory test or examination exactly as reported in the clinical document "
                    "(e.g. creatinine, potassium). "
                    "Always populate this field when a laboratory value is extracted."
    )
    lab_value: str = Field(default=None,
        description="Laboratory result including numerical value only. "
                    "Return the value exactly as reported in the source text."
                    "Set to null if no numerical value is reported."
    )

    lab_unit: str = Field(default=None,
        description="Measurement unit associated with the laboratory result. "
                    "Return the unit exactly as reported in the source text. "
                    "Set to null if no unit is reported."
    )

    lab_date: str = Field(default=None,
        description="Date associated with the laboratory result. "
                    "Use dd.mm.yyyy format when possible. "
                    "Set to null if no timestamp is reported."
    )
    notes: str = Field(default=None,
        description="Extra information about the laboratory result. Set to null if not reported."
    )

class HFrEFPatientSchema(BaseModel):
    demographic_info: List[str] = Field(
        default=None,
        description="Demographic information explicitly reported in the clinical document. "
                    "Extract relevant patient demographic characteristics such as age, sex, ethnicity, "
                    "race, or other demographic descriptors when available. "
                    "Return each demographic item as a separate string. "
                    "Set to null if no demographic information is reported."
    )

    drug_intolerances: List[str] = Field(
        default=None,
        description="Drug intolerances explicitly mentioned in the clinical letter. "
                    "Extract all medications reported as not tolerated by the patient. "
                    "Return each drug intolerance as a separate string using concise medical terminology. "
                    "Do not infer information that is not explicitly stated. "
                    "Avoid duplicates and set to null if no drug intolerances are reported."
    )

    lab_values: List[LabEntry] = Field(
        default=None,
        description="Extract all medical measurements, vital signs, and laboratory results reported in the clinical document. "
                    "Specifically, you must include laboratory tests (such as creatinine, BNP) and physiological parameters "
                    "like Heart Rate (HR), Left Ventricular Ejection Fraction (LVEF), Systolic Blood Pressure (SBP), and Diastolic Blood Pressure (DBP). "
                    "For each measurement, extract the test or parameter name, its numerical value, measurement unit, and associated date/time when available. "
                    "Return one LabEntry object per result. "
                    "CRITICAL RULE: If multiple measurements or values are reported for the same test or parameter across different times or dates, extract ONLY the latest (most recent) measurement available in the text. "
                    "Do not infer missing values or timestamps. "
                    "Set to null if no matching data are reported."
    )

    comorbidities: List[str] = Field(
        default=None,
        description="List of Heart Failure comorbidities explicitly mentioned in the clinical letter. "
                    "Heart Failure (HF) is the target disease of this study and MUST NEVER be returned in this field."
                    "Exclude any diagnosis, condition, syndrome, phenotype, subtype, stage, severity descriptor, or clinical expression that represents, contains, modifies, qualifies, or refers to Heart Failure."
                    "CRITICAL: If a candidate extraction contains the words 'heart failure', 'cardiac failure', 'compensation', 'decompensation', or an HF abbreviation (HF, HFrEF, HFmrEF, HFpEF, HFimpEF), discard it."
                    "Return each comorbidity as a separate string. "
                    "Do not duplicate the same underlying disease using different clinical phenotypes, stages, subtypes, or temporal evolutions. "
                    "If a diagnosis represents an improved, recovered, chronic, acute, decompensated, compensated, or otherwise qualified form of the same disease, return only the diagnosis exactly as reported in the letter and do not add the parent or previous condition unless it is also explicitly listed as a separate active diagnosis. "
                    "Do not infer conditions that are not explicitly stated. "
                    "Set to null if no comorbidities are reported."
    )

    disorders_and_conditions: List[str] = Field(
        default=None,
        description="Extract only clinically recognized diagnoses, diseases, syndromes, pathological conditions, or symptoms explicitly reported in the letter that do NOT qualify as comorbidities. "
                    "AVOID findings already classified as comorbidities. "
                    "Do not extract acute, transient, or resolved events that are clearly described as no longer clinically relevant, unless they are explicitly reported as part of the patient's current comorbidity profile."
                    "Heart Failure (HF) is the target disease of this study and MUST NEVER be returned in this field."
                    "Exclude any diagnosis, condition, syndrome, phenotype, subtype, stage, severity descriptor, or clinical expression that represents, contains, modifies, qualifies, or refers to Heart Failure (HF)."
                    "CRITICAL: If a candidate extraction contains the words 'heart failure', 'cardiac failure', 'compensation', 'decompensation', or an HF abbreviation (HF, HFrEF, HFmrEF, HFpEF, HFimpEF), DISCARD it."
                    "Do NOT extract quantitative or functional cardiac measurements, indices, or parameters used to describe disease severity or cardiac function, including but not limited to: left ventricular ejection fraction (LVEF / EF / FE), ventricular volumes, wall motion indices, strain measures, or similar cardiological functional metrics."
                    "Return each disorder or condition as a separate string using concise medical terminology."
                    "Do not infer information that is not explicitly stated."
                    "Avoid duplicates and set to null if no disorders or conditions are reported."
    )

    # Clinical criteria:

    worsening_heart_failure: bool = Field(
        default=False,
        description="Whether the patient is currently experiencing worsening heart failure. "
                    "Return True if the patient has been hospitalized or had an unplanned outpatient visit for heart failure (and treatment with either intravenous diuretics, vasodilators or inotropes) within the last 6 months. "
                    "Return True if explicitly reported. "
                    "Return False if explicitly denied. "
                    "Set to False if the information is not available."
    )

    loop_diuretic: bool = Field(
        default=False,
        description="Whether the patient is treated with a loop diuretic BEFORE the time of the visit. "
                    "Consider ONLY drugs the patient was ALREADY taking at the time of the visit. "
                    "Do NOT consider the medication regimen prescribed after the visit. "
                    "Return True if loop diuretic therapy is explicitly reported. "
                    "Return False if explicitly denied. "
                    "Set to False if the information is not available."
    )

    peripheral_edema: bool = Field(
        default=False,
        description="Presence of peripheral edema reported by the patient or identified during physical examination. "
                    "Return True only if edema is explicitly described. "
                    "Return False if explicitly absent. "
                    "Set to False if not reported."
    )

    hfrEF_90_days: bool = Field(
        default=False,
        description="Whether the patient had a diagnosis of heart failure with reduced ejection fraction (HFrEF) "
                    "at least 3 months prior to enrollment or evaluation. "
                    "HFrEF is defined as left ventricular ejection fraction (LVEF) less than or equal to 40% "
                    "associated with signs and symptoms of heart failure such as dyspnea or NYHA class II-IV. "
                    "Return True only if this condition is explicitly documented. "
                    "Return False if explicitly denied. "
                    "Set to null if the information is unavailable."
    )

    comfort_care_only: bool = Field(
        default=False,
        description="Whether the patient is currently receiving comfort care only. "
                    "Return True if the clinical document explicitly states that the patient is under comfort care, palliative care, hospice care, or has a do-not-resuscitate (DNR) order indicating a focus on comfort measures rather than curative treatment. "
                    "Return False if explicitly denied or if the patient is enrolled in a hospice program. "
                    "Set to False if the information is not available."
    )

    hfrEF_etiology: HFrEF_Etiology = Field(
        default=None,
        description="Etiology or underlying cause of heart failure with reduced ejection fraction (HFrEF) "
                    "as explicitly reported in the clinical document. "
                    "Select one from the predefined list: ISCHEMIC / NON_ISCHEMIC"
                    "Set to null if the etiology is not reported."
    )

    discharge_date: str = Field(
        default=None,
        description="Date of the most recent hospital discharge related to heart failure. "
                    "Use dd.mm.yyyy format when possible. "
                    "Set to null if no discharge date is reported or if the patient has not been hospitalized for heart failure."
    )

    # End of clinical criteria

    current_therapy: List[TherapyEntry] = Field(
        default=None,
        description="Current pharmacological therapies reported in the clinical letter. "
                    "List all drugs the patient was already taking at the time of the visit. "
                    "Extract medications that the patient is currently taking or receiving at the time of the clinical assessment, visit, or hospital admission. "
                    "Include chronic home medications and ongoing treatments explicitly described as current therapy. "
                    "Exclude discharge medications unless they are explicitly stated to be part of the current regimen at the time of assessment. "
                    "For each therapy include drug name, dosage information, dose modifications, uptitration status, relevant dates, and additional notes when available. "
                    "Return one TherapyEntry object per medication. "
                    "Set to null if no current therapy is reported. "
    )

    discharge_therapy: List[TherapyEntry] = Field(
        default=None,
        description="Extract the COMPLETE discharge medication regimen the patient must follow after the current visit. "
                    "CRITICAL: You must include BOTH newly prescribed medications AND pre-existing/chronic medications that the patient is instructed to continue taking. "
                    "Scan the entire text (including 'Medical History', 'Anamnesis', 'Home Therapy', and 'Discharge') to build the full list. "
                    "For each therapy include drug name, dosage information, dose modifications, "
                    "uptitration status, start dates if available, and additional notes when available. "
                    "For each therapy include drug name, dosage information, dose modifications, uptitration status, relevant dates, and additional notes when available. "
                    "Return one TherapyEntry object per medication. "
                    "Set to null ONLY if the text explicitly states no medications are prescribed at discharge. "
    )

# Original variables with text span

In [ ]:
T = TypeVar("T")

class Extracted(BaseModel, Generic[T]):
    value: T
    source_text: str = Field(
        default=None,
        description="Original span from the source document from which the value was extracted. Set to null if the value was missing."
    )

class TherapyEntry(BaseModel):
    drug_name: Extracted[str] = Field(default=None,
        description="Drug name as reported in the source, exactly as written. "
                    "Always populate this field."
    )
    generic_drug_name: Extracted[str] = Field(default=None,
        description="Drug generic name or ATC code, only if clearly identifiable as such. "
                    "Return the output translated in ENGLISH"
                    "Exclude the dose from the output. "
                    "Set to null if not reported or not distinguishable."
    )
    brand_drug_name: Extracted[str] = Field(default=None,
        description="Drug brand name, only if clearly identifiable as such. "
                    "Set to null if not reported or not distinguishable."
    )
    active_ingredient: Extracted[str] = Field(default=None,
        description="Extract the dose strength and its mass/activity unit of measurement (e.g., mg, mcg, µg, UI) for the medication, as explicitly reported in the text."
                    "Formatting rules:"
                    "- For single active ingredients: Output MUST be strictly in the format <number> <unit> (e.g., 20 mg, 50 mcg, 1000 UI)."
                    "- For combination drugs (e.g., 97mg+103mg or 97mg/103mg): Join the numbers with a slash and follow with a single space and unit, formatted as <number>/<number> <unit> (e.g., 97/103 mg). If units differ, maintain their respective units (e.g., 50 mcg/10 mg)."
                    "Ignore pharmaceutical forms, tablet/pill counts, administration timing, and frequencies."
    )

    dose_units_number: Extracted[float] = Field(default=None,
        description="Calculate ONLY the total count of single physical dose units (e.g., tablets, drops, vials, puffs) to be taken across the ENTIRE 24-HOUR DAY, expressed as a single decimal or integer number."
                    "Rules:"
                    "MULTIPLE TIMESTAMPS FOR ONE DOSE: If a dose is linked to multiple hours separated by hyphens, commas, or slashes (e.g., '1 cpr at 8-20', '1/2 cpr hour 8,14,20'), multiply the unit dose by the number of listed hours (e.g., 1 tablet at 8 and 20 = 1 * 2 = 2)."
                    "- SUM MULTIPLE EVENTS: If multiple individual doses, timestamps, or times of day are listed (e.g., '1 tablet at 8:00 and 1 tablet at 20:00', '1 tab morning and 1 tab night'), SUM all individual physical units together (1 + 1 = 2)."
                    "- FREQUENCY MULTIPLICATION: If an explicit daily frequency is stated (e.g., '1/2 tablet twice a day', '1 puff 3 times daily'), multiply the unit dose by the frequency (0.5 * 2 = 1)."
                    "- TIMESTAMPS VS COUNTS: Do NOT use the hour value as a dose (e.g., in 'at 14:00', 14 is the hour, NOT 14 tablets). However, DO count the physical dose associated with that time."
                    "- CONVERSIONS: Convert words, fractions, and partial units to decimals (e.g., 1/2 -> 0.5, half -> 0.5, one -> 1, two -> 2)."
                    "- IGNORE DOSAGE STRENGTHS: Ignore mass/volume units (e.g., mg, mcg, ml) and focus ONLY on physical unit counts."
                    "- SINGLE EVENT: If only a single administration is described for the day with no other times mentioned (e.g., 1/2 tablet at 14), the daily total is that single dose (0.5)."

                    "Examples: "
                    "- '1 cpr at 8-20' -> 2"
                    "- '1/2 cpr at 8-14-20' -> 1.5"
                    "- '1/2 tablet twice a day' -> 1"
                    "- '1 tablet morning, 1/2 tablet at 14:00, 1 tablet at night' -> 2.5"
                    "- 'Bisoprolol 5mg 1/2 tablet at 8:00' -> 0.5"

                    "Output MUST be ONLY the single calculated number or null. No text, explanation, or unit labels."
    )
    
    drug_class: Extracted[Drug_Class] = Field(
        default=None,
        description="Analyze the drug name following these steps:"
                    "1. Identify the active ingredient."
                    "2. Assign the corresponding class from the predefined list: ACE_I / ARB / ARNI / BETA_BLOCKER / MRA / SGLT2_INHIBITOR / LOOP_DIURETIC."
                    "Examples:"
                    "- ACE_I: enalapril, ramipril, lisinopril"
                    "- ARB: losartan, valsartan, candesartan"
                    "- ARNI: sacubitril/valsartan"
                    "- BETA_BLOCKER: bisoprolol, carvedilol, metoprolol"
                    "- MRA: spironolactone, eplerenone, canrenone (and its potassium salt, "
                    "  canrenoate)"
                    "- SGLT2_INHIBITOR: dapagliflozin, empagliflozin"
                    "- LOOP_DIURETIC: furosemide, torasemide, bumetanide"
                    "3. If the active ingredient does not match any list, set to null."
    )
    notes: Extracted[str] = Field(default=None,
        description="Extra information about the specified drug. "
                    "Specify any relevant details such as dose modifications, uptitration status, dates of therapy initiation or changes, or other clinically relevant information. "
                    "Set to null if not reported."
    )

class LabEntry(BaseModel):
    lab_exam: Extracted[str] = Field(default=None,
        description="Name of the laboratory test or examination exactly as reported in the clinical document "
                    "(e.g. creatinine, potassium). "
                    "Always populate this field when a laboratory value is extracted."
    )
    lab_value: Extracted[str] = Field(default=None,
        description="Laboratory result including numerical value only. "
                    "Return the value exactly as reported in the source text."
                    "Set to null if no numerical value is reported."
    )

    lab_unit: Extracted[str] = Field(default=None,
        description="Measurement unit associated with the laboratory result. "
                    "Return the unit exactly as reported in the source text. "
                    "Set to null if no unit is reported."
    )

    lab_date: Extracted[str] = Field(default=None,
        description="Date associated with the laboratory result. "
                    "Use dd.mm.yyyy format when possible. "
                    "Set to null if no timestamp is reported."
    )
    notes: Extracted[str] = Field(default=None,
        description="Extra information about the laboratory result. Set to null if not reported."
    )

class HFrEFPatientSchema(BaseModel):
    demographic_info: List[Extracted[str]] = Field(
        default=None,
        description="Demographic information explicitly reported in the clinical document. "
                    "Extract relevant patient demographic characteristics such as age, sex, ethnicity, "
                    "race, or other demographic descriptors when available. "
                    "Return each demographic item as a separate string. "
                    "Set to null if no demographic information is reported."
    )

    drug_intolerances: List[Extracted[str]] = Field(
        default=None,
        description="Drug intolerances explicitly mentioned in the clinical letter. "
                    "Extract all medications reported as not tolerated by the patient. "
                    "Return each drug intolerance as a separate string using concise medical terminology. "
                    "Do not infer information that is not explicitly stated. "
                    "Avoid duplicates and set to null if no drug intolerances are reported."
    )

    lab_values: List[Extracted[LabEntry]] = Field(
        default=None,
        description="Extract all medical measurements, vital signs, and laboratory results reported in the clinical document. "
                    "Specifically, you must include laboratory tests (such as creatinine, BNP) and physiological parameters "
                    "like Heart Rate (HR), Left Ventricular Ejection Fraction (LVEF), Systolic Blood Pressure (SBP), and Diastolic Blood Pressure (DBP). "
                    "For each measurement, extract the test or parameter name, its numerical value, measurement unit, and associated date/time when available. "
                    "Return one LabEntry object per result. "
                    "CRITICAL RULE: If multiple measurements or values are reported for the same test or parameter across different times or dates, extract ONLY the latest (most recent) measurement available in the text. "
                    "Do not infer missing values or timestamps. "
                    "Set to null if no matching data are reported."
    )

    comorbidities: List[Extracted[str]] = Field(
        default=None,
        description="List of Heart Failure comorbidities explicitly mentioned in the clinical letter. "
                    "Heart Failure (HF) is the target disease of this study and MUST NEVER be returned in this field."
                    "Exclude any diagnosis, condition, syndrome, phenotype, subtype, stage, severity descriptor, or clinical expression that represents, contains, modifies, qualifies, or refers to Heart Failure."
                    "CRITICAL: If a candidate extraction contains the words 'heart failure', 'cardiac failure', 'compensation', 'decompensation', or an HF abbreviation (HF, HFrEF, HFmrEF, HFpEF, HFimpEF), discard it."
                    "Return each comorbidity as a separate string. "
                    "Do not duplicate the same underlying disease using different clinical phenotypes, stages, subtypes, or temporal evolutions. "
                    "If a diagnosis represents an improved, recovered, chronic, acute, decompensated, compensated, or otherwise qualified form of the same disease, return only the diagnosis exactly as reported in the letter and do not add the parent or previous condition unless it is also explicitly listed as a separate active diagnosis. "
                    "Do not infer conditions that are not explicitly stated. "
                    "Set to null if no comorbidities are reported."
    )

    disorders_and_conditions: List[Extracted[str]] = Field(
        default=None,
        description="Extract only clinically recognized diagnoses, diseases, syndromes, pathological conditions, or symptoms explicitly reported in the letter that do NOT qualify as comorbidities. "
                    "AVOID findings already classified as comorbidities. "
                    "Do not extract acute, transient, or resolved events that are clearly described as no longer clinically relevant, unless they are explicitly reported as part of the patient's current comorbidity profile."
                    "Heart Failure (HF) is the target disease of this study and MUST NEVER be returned in this field."
                    "Exclude any diagnosis, condition, syndrome, phenotype, subtype, stage, severity descriptor, or clinical expression that represents, contains, modifies, qualifies, or refers to Heart Failure (HF)."
                    "CRITICAL: If a candidate extraction contains the words 'heart failure', 'cardiac failure', 'compensation', 'decompensation', or an HF abbreviation (HF, HFrEF, HFmrEF, HFpEF, HFimpEF), DISCARD it."
                    "Do NOT extract quantitative or functional cardiac measurements, indices, or parameters used to describe disease severity or cardiac function, including but not limited to: left ventricular ejection fraction (LVEF / EF / FE), ventricular volumes, wall motion indices, strain measures, or similar cardiological functional metrics."
                    "Return each disorder or condition as a separate string using concise medical terminology."
                    "Do not infer information that is not explicitly stated."
                    "Avoid duplicates and set to null if no disorders or conditions are reported."
    )

    # Clinical criteria:

    worsening_heart_failure: Extracted[bool] = Field(
        default=False,
        description="Whether the patient is currently experiencing worsening heart failure. "
                    "Return True if the patient has been hospitalized or had an unplanned outpatient visit for heart failure (and treatment with either intravenous diuretics, vasodilators or inotropes) within the last 6 months. "
                    "Return True if explicitly reported. "
                    "Return False if explicitly denied. "
                    "Set to False if the information is not available."
    )

    loop_diuretic: Extracted[bool] = Field(
        default=False,
        description="Whether the patient is treated with a loop diuretic BEFORE the time of the visit. "
                    "Consider ONLY drugs the patient was ALREADY taking at the time of the visit. "
                    "Do NOT consider the medication regimen prescribed after the visit. "
                    "Return True if loop diuretic therapy is explicitly reported. "
                    "Return False if explicitly denied. "
                    "Set to False if the information is not available."
    )

    peripheral_edema: Extracted[bool] = Field(
        default=False,
        description="Presence of peripheral edema reported by the patient or identified during physical examination. "
                    "Return True only if edema is explicitly described. "
                    "Return False if explicitly absent. "
                    "Set to False if not reported."
    )

    hfrEF_90_days: Extracted[bool] = Field(
        default=False,
        description="Whether the patient had a diagnosis of heart failure with reduced ejection fraction (HFrEF) "
                    "at least 3 months prior to enrollment or evaluation. "
                    "HFrEF is defined as left ventricular ejection fraction (LVEF) less than or equal to 40% "
                    "associated with signs and symptoms of heart failure such as dyspnea or NYHA class II-IV. "
                    "Return True only if this condition is explicitly documented. "
                    "Return False if explicitly denied. "
                    "Set to null if the information is unavailable."
    )

    comfort_care_only: Extracted[bool] = Field(
        default=False,
        description="Whether the patient is currently receiving comfort care only. "
                    "Return True if the clinical document explicitly states that the patient is under comfort care, palliative care, hospice care, or has a do-not-resuscitate (DNR) order indicating a focus on comfort measures rather than curative treatment. "
                    "Return False if explicitly denied or if the patient is enrolled in a hospice program. "
                    "Set to False if the information is not available."
    )

    hfrEF_etiology: Extracted[HFrEF_Etiology] = Field(
        default=None,
        description="Etiology or underlying cause of heart failure with reduced ejection fraction (HFrEF) "
                    "as explicitly reported in the clinical document. "
                    "Select one from the predefined list: ISCHEMIC / NON_ISCHEMIC"
                    "Set to null if the etiology is not reported."
    )

    discharge_date: Extracted[str] = Field(
        default=None,
        description="Date of the most recent hospital discharge related to heart failure. "
                    "Use dd.mm.yyyy format when possible. "
                    "Set to null if no discharge date is reported or if the patient has not been hospitalized for heart failure."
    )

    # End of clinical criteria

    current_therapy: List[Extracted[TherapyEntry]] = Field(
        default=None,
        description="Current pharmacological therapies reported in the clinical letter. "
                    "List all drugs the patient was already taking at the time of the visit. "
                    "Extract medications that the patient is currently taking or receiving at the time of the clinical assessment, visit, or hospital admission. "
                    "Include chronic home medications and ongoing treatments explicitly described as current therapy. "
                    "Exclude discharge medications unless they are explicitly stated to be part of the current regimen at the time of assessment. "
                    "For each therapy include drug name, dosage information, dose modifications, uptitration status, relevant dates, and additional notes when available. "
                    "Return one TherapyEntry object per medication. "
                    "Set to null if no current therapy is reported. "
    )

    discharge_therapy: List[Extracted[TherapyEntry]] = Field(
        default=None,
        description="Extract the COMPLETE discharge medication regimen the patient must follow after the current visit. "
                    "CRITICAL: You must include BOTH newly prescribed medications AND pre-existing/chronic medications that the patient is instructed to continue taking. "
                    "Scan the entire text (including 'Medical History', 'Anamnesis', 'Home Therapy', and 'Discharge') to build the full list. "
                    "For each therapy include drug name, dosage information, dose modifications, "
                    "uptitration status, start dates if available, and additional notes when available. "
                    "For each therapy include drug name, dosage information, dose modifications, uptitration status, relevant dates, and additional notes when available. "
                    "Return one TherapyEntry object per medication. "
                    "Set to null ONLY if the text explicitly states no medications are prescribed at discharge. "
    )

# LLM code

In [ ]:
with open("system_prompt.txt", "r", encoding="utf-8") as f:
    system_prompt = f.read()

def read_patient_file(patient_file_name):
    # Ricaviamo l'estensione del file in minuscolo (es: '.docx', '.txt', '.rtf')
    _, extension = os.path.splitext(patient_file_name.lower())
    
    # --- GESTIONE FILE .TXT ---
    if extension == '.txt':
        # Usiamo errors='ignore' o 'replace' per evitare blocchi dovuti a caratteri speciali o accenti
        with open(patient_file_name, 'r', encoding='utf-8', errors='replace') as f:
            return f.read()
            
    # --- GESTIONE FILE .RTF ---
    elif extension == '.rtf':
        with open(patient_file_name, 'r', encoding='utf-8', errors='replace') as f:
            rtf_content = f.read()
            # Converte l'RTF in testo semplice privo di codice di formattazione
            return rtf_to_text(rtf_content)
            
    # --- GESTIONE FILE .DOCX ---
    elif extension == '.docx':
        doc = Document(patient_file_name)
        content = []

        for element in doc.element.body:
            # Paragraph
            if element.tag == qn("w:p"):
                text = "".join(node.text for node in element.iter(qn("w:t")) if node.text)
                if text.strip():
                    content.append(text)

            # Table
            elif element.tag == qn("w:tbl"):
                for row in element.iter(qn("w:tr")):
                    cells = []
                    for cell in row.iter(qn("w:tc")):
                        cell_text = "".join(node.text for node in cell.iter(qn("w:t")) if node.text)
                        cells.append(cell_text.strip())
                    row_text = "\t".join(cells)
                    if row_text.strip():
                        content.append(row_text)

        return "\n".join(content)

In [ ]:
def fix_control_chars(s: str) -> str:
    # sostituisce TAB reali con escape JSON valido
    s = s.replace("\t", "\\t")

    # (opzionale) rimuove altri controlli problematici
    s = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", s)

    return s

def parse_llm(content):
    content = content.replace("```json", "").replace("```", "").strip()

    # FIX TAB (questo è il punto chiave)
    content = fix_control_chars(content)

    return json.loads(content)

## Local LLM

In [ ]:
def call_llm_port(model, prompt, response_format):
    # Headers for authentication
    headers = {
        'Content-Type': 'application/json'
    }

    from dotenv import load_dotenv
    load_dotenv(override=True)
    url = os.getenv("OLLAMA_URL", "http://localhost:11434/api/chat")

    payload = {
        "model": model,
        "stream": False,
        "messages": prompt,
        "response_format": {
            "type": "json_schema",
            "json_schema": response_format.model_json_schema()
        }
    }

    # Convert the payload to a JSON string
    payload_json = json.dumps(payload)

    # Make the POST request
    response = requests.post(url, headers=headers, data=payload_json, proxies={"http": None, "https": None})

    # Checking if the request was successful
    response.raise_for_status()  # Raise an exception for HTTP errors

    # Processing the response data (assuming JSON response)
    data = response.json()

    return data

In [ ]:
HFrEF_Letter_1=[]
#Tested models: gpt-oss:120b-cloud, gemma4:31b-cloud
model_names=["gpt-oss:120b-cloud","gemma4:31b-cloud"]

os.makedirs("HFrEF_results", exist_ok=True)  # assicura che la cartella esista

for short_file_name in HFrEF_Letter_1:
    text = read_patient_file(f"HFrEF letters/{short_file_name}.docx")
    
    for model_name in model_names:
        print(f"Processing {short_file_name} with {model_name}")

        messages= [{ "role": "system", "content": Template(system_prompt).substitute(output_schema=json.dumps(HFrEFPatientSchema.model_json_schema()), indent=2) },
                    { "role": "user", "content": text }]

        try:
            answer = call_llm_port(model_name, messages, HFrEFPatientSchema)
            data = parse_llm(answer["message"]["content"])
        except Exception as e:
            print(f"  Errore con {model_name} su {short_file_name}: {e}")
            continue  # passa al modello/lettera successivo senza bloccare tutto

        model_name_safe = model_name.replace(":", "_")
        file_path = f"HFrEF_results/{short_file_name}_{model_name_safe}_with_span.json"

        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4, ensure_ascii=False)

## AWS LLM

In [ ]:
import boto3
from dotenv import load_dotenv
from botocore.exceptions import ClientError
import logging
logger=logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO,format="%(levelname)s:%(message)s")
load_dotenv(override=True)

def get_bedrock_client(region_name: str = "us-west-2"):
    """
        Get Bedrock client.
        Args:
            region_name (str): The AWS region to invoke the model on
        Returns:
            bedrock_wrapper.client.BedrockClient: The bedrock client
        """
    # Create an Amazon Bedrock client
    return boto3.client(service_name="bedrock-runtime", region_name=region_name)


def call_bedrock(system_prompt: str, user_prompt: str, model_id: str, response_format, client) -> dict:
    try:
        response = client.converse(
            modelId=model_id,
            messages=[{"role": "user", "content": [{"text": user_prompt}]}],
            system=[{"text": system_prompt}])
    except ClientError as err:
        message = err.response["Error"]["Message"]
        logger.error("A client error occurred: %s", message)
        raise RuntimeError(f"Chiamata Bedrock fallita: {message}") from err
    else:
        return response['output']['message']

In [ ]:
client=get_bedrock_client()

In [ ]:
HFrEF_Letter_1=[]
#deepseek.v3.2, us.anthropic.claude-sonnet-4-6, qwen.qwen3-32b-v1:0
model_names=["deepseek.v3.2","qwen.qwen3-32b-v1:0","us.anthropic.claude-sonnet-4-6"]

os.makedirs("HFrEF_results", exist_ok=True)  # assicura che la cartella esista

for short_file_name in HFrEF_Letter_1:
    text = read_patient_file(f"HFrEF letters/{short_file_name}.docx")
    
    for model_name in model_names:
        print(f"Processing {short_file_name} with {model_name}")

        messages= [{ "role": "system", "content": Template(system_prompt).substitute(output_schema=json.dumps(HFrEFPatientSchema.model_json_schema()), indent=2) },
        { "role": "user", "content": text }]

        try:
            answer = call_bedrock(Template(system_prompt).substitute(output_schema=json.dumps(HFrEFPatientSchema.model_json_schema()), indent=2),
                                  text, model_name, HFrEFPatientSchema,client)
            data=parse_llm(answer["content"][0]["text"])
        except Exception as e:
            print(f"  Errore con {model_name} su {short_file_name}: {e}")
            continue  # passa al modello/lettera successivo senza bloccare tutto

        model_name_safe = model_name.replace(":", "_")
        file_path = f"HFrEF_results/{short_file_name}_{model_name_safe}_with_span.json"

        with open(file_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=4, ensure_ascii=False)